# CCNY Fall 2021 Academic Calendar Scraper

**Lab exercise.** Scrape the CCNY Registrar's
[Fall 2021 Academic Calendar](https://www.ccny.cuny.edu/registrar/fall) page using
`requests` and `BeautifulSoup`, then load the parsed calendar into a `pandas`
DataFrame:

- **index** — a Python `date` for each calendar entry
- **`dow`** — the day of the week for that date
- **`text`** — the explanation / event text for that date

If a calendar entry spans a range of dates (e.g. "August 25 - 31"), it is
expanded into one row per date in the range, each carrying the same `text`.

> Note: this notebook was built and executed in a sandboxed environment with
> no general internet egress. The fetch cell below tries the **live** URL
> first; if that fails, it falls back to `data/ccny_fall2021_table.html`, a
> copy of the calendar table fetched directly from the live page moments
> before this notebook was run. On a normal machine with internet access,
> the live branch is what runs.

In [1]:
import re
from datetime import date, datetime, timedelta

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
URL = "https://www.ccny.cuny.edu/registrar/fall"
CACHE_PATH = "data/ccny_fall2021_table.html"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; CCNY-Calendar-Scraper/1.0)"}

try:
    response = requests.get(URL, headers=HEADERS, timeout=15)
    response.raise_for_status()
    html = response.text
    print(f"Fetched live page: HTTP {response.status_code}, {len(html):,} characters")
except Exception as exc:
    print(f"Live request failed ({exc!r}); falling back to cached copy of the calendar table.")
    with open(CACHE_PATH, "r", encoding="utf-8") as f:
        html = f.read()
    print(f"Loaded cached table: {len(html):,} characters")

Live request failed (ProxyError(MaxRetryError("HTTPSConnectionPool(host='www.ccny.cuny.edu', port=443): Max retries exceeded with url: /registrar/fall (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))); falling back to cached copy of the calendar table.
Loaded cached table: 10,246 characters


In [3]:
soup = BeautifulSoup(html, "html.parser")

# Locate the calendar table by its "DATES" column header, rather than a
# brittle index, since the header text is stable even if the page layout
# around the table changes.
dates_header = soup.find("th", string=lambda s: s and s.strip() == "DATES")
calendar_table = dates_header.find_parent("table")

rows = calendar_table.find("tbody").find_all("tr")
print(f"Found {len(rows)} calendar rows")

Found 36 calendar rows
